In [ ]:
# 1. 환경 설정 (Drive 마운트 + 패키지 설치 + 캐시를 Drive로)
import os, subprocess, sys, shutil

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")
CACHE_DIR  = os.path.join(DRIVE_DIR, ".cache")

# Resume ckpt는 로컬에 저장 (빠르고 디스크 절약, 세션 종료 시 사라짐)
LOCAL_CKPT = "/content/dacon/local_ckpt"

for d in [
    DRIVE_DIR,
    os.path.join(DRIVE_DIR, "checkpoints"),
    os.path.join(DRIVE_DIR, "submissions"),
    CACHE_DIR,
    WORK_DIR,
    LOCAL_CKPT,
]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_DIR, "huggingface")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_DIR, "torch")
os.environ["XDG_CACHE_HOME"] = CACHE_DIR
os.environ["PIP_CACHE_DIR"] = os.path.join(CACHE_DIR, "pip")
os.environ["TMPDIR"] = os.path.join(CACHE_DIR, "tmp")
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

# Resume ckpt → 로컬, best model → Drive (checkpoints/ symlink)
os.environ["DACON_LOCAL_CKPT"] = LOCAL_CKPT

import torch
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu} ({vram:.1f} GB)")
else:
    print("GPU 없음! 런타임 > 런타임 유형 변경 > GPU 선택")

free = shutil.disk_usage('/content').free / 1e9
print(f"로컬 디스크 여유: {free:.1f} GB")
print(f"Resume ckpt 경로: {LOCAL_CKPT} (로컬, 세션 종료 시 삭제)")
print(f"Best model 경로: {DRIVE_DIR}/checkpoints/ (Drive, 영구)")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm>=1.0.20", "albumentations>=1.4.0",
    "opencv-python-headless", "scikit-learn", "pandas", "tqdm",
])
print("패키지 설치 완료")

In [ ]:
# 2. 소스 파일 복사 + 체크포인트 설정 (로컬 저장 -> Drive 동기화)
import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/dacon"
WORK_DIR  = "/content/dacon"
DRIVE_CKPT = os.path.join(DRIVE_DIR, "checkpoints")
LOCAL_CKPT_BEST = os.path.join(WORK_DIR, "checkpoints")   # best model 로컬 저장
DRIVE_SUB  = os.path.join(DRIVE_DIR, "submissions")

# --- 소스 파일 복사 ---
SRC_FILES = ["models.py", "datasets.py", "train.py", "train_v2.py", "inference.py", "inference_v2.py"]
for f in SRC_FILES:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join(WORK_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  [OK] {f}")
    else:
        print(f"  [MISSING] {f} - Drive에 업로드 필요: {DRIVE_DIR}/")

# --- 체크포인트: 로컬 디렉토리 (Drive symlink 안 함 = 빠른 저장) ---
os.makedirs(LOCAL_CKPT_BEST, exist_ok=True)

# --- submissions: Drive 직접 (작은 CSV라 느려도 OK) ---
os.makedirs(DRIVE_SUB, exist_ok=True)
sub_link = os.path.join(WORK_DIR, "submissions")
if os.path.islink(sub_link):
    os.unlink(sub_link)
elif os.path.isdir(sub_link):
    shutil.rmtree(sub_link)
os.symlink(DRIVE_SUB, sub_link)
print(f"  submissions/ -> Drive (영구)")

# --- Drive에 기존 best 가중치 있으면 로컬로 복사 (resume용) ---
os.makedirs(DRIVE_CKPT, exist_ok=True)
copied = 0
for f in sorted(os.listdir(DRIVE_CKPT)):
    if not f.endswith('.pth'):
        continue
    src = os.path.join(DRIVE_CKPT, f)
    dst = os.path.join(LOCAL_CKPT_BEST, f)
    if not os.path.exists(dst):
        mb = os.path.getsize(src) / 1e6
        print(f"  <- Drive->로컬: {f} ({mb:.0f} MB)")
        shutil.copy2(src, dst)
        copied += 1
if copied == 0:
    print("  Drive 체크포인트 없음 또는 이미 로컬에 존재")

# --- 동기화 헬퍼 함수 (학습 후 호출) ---
def sync_best_to_drive(backbone=None):
    """로컬 best 가중치 -> Drive 복사 (ckpt 제외, best만)"""
    count = 0
    for f in sorted(os.listdir(LOCAL_CKPT_BEST)):
        if not f.endswith('.pth'):
            continue
        if '_ckpt' in f:
            continue  # resume ckpt는 로컬에만
        if backbone and not f.startswith(backbone):
            continue
        src = os.path.join(LOCAL_CKPT_BEST, f)
        dst = os.path.join(DRIVE_CKPT, f)
        src_size = os.path.getsize(src)
        # 이미 동일 크기면 스킵
        if os.path.exists(dst) and os.path.getsize(dst) == src_size:
            continue
        mb = src_size / 1e6
        print(f"  -> Drive: {f} ({mb:.0f} MB)")
        shutil.copy2(src, dst)
        count += 1
    if count == 0:
        print("  동기화할 새 가중치 없음")
    else:
        print(f"  {count}개 가중치 Drive 동기화 완료")

print(f"\n체크포인트 저장: 로컬 ({LOCAL_CKPT_BEST})")
print("학습 후 sync_best_to_drive() 호출하면 Drive에 복사됨")
print(f"Drive 체크포인트: {DRIVE_CKPT}")

In [ ]:
# 2.5 코랩 경쟁 모드 패치 (로컬 파일 영향 없음)
# 더 강한 증강 + CenterCrop TTA를 코랩 런타임에만 적용

from pathlib import Path

path = Path("/content/dacon/datasets.py")
text = path.read_text(encoding="utf-8")

old_train = '''def get_train_transforms(img_size=384):
    \"\"\"경쟁급 최대 증강\"\"\"
    return A.Compose([
        A.RandomResizedCrop(
            (img_size, img_size), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(-0.05, 0.05), scale=(0.93, 1.07),
            rotate=(-10, 10), p=0.5),
        A.Perspective(scale=(0.02, 0.06), p=0.3),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.HueSaturationValue(
                hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=1.0),
            A.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=1.0),
        ], p=0.8),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
        ], p=0.2),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.10), p=1.0),
            A.ISONoise(p=1.0),
        ], p=0.2),
        A.RandomShadow(p=0.15),
        A.RandomFog(fog_coef_range=(0.1, 0.35), p=0.1),
        A.CoarseDropout(
            num_holes_range=(1, 6),
            hole_height_range=(16, 40), hole_width_range=(16, 40), p=0.25),
        A.ToGray(p=0.05),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])'''

new_train = '''def get_train_transforms(img_size=384):
    \"\"\"코랩 경쟁 모드용 강화 증강\"\"\"
    return A.Compose([
        A.RandomResizedCrop(
            (img_size, img_size), scale=(0.72, 1.0), ratio=(0.88, 1.12), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(-0.08, 0.08), scale=(0.90, 1.10),
            rotate=(-12, 12), p=0.6),
        A.Perspective(scale=(0.03, 0.08), p=0.35),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.25, contrast_limit=0.25, p=1.0),
            A.HueSaturationValue(
                hue_shift_limit=12, sat_shift_limit=18, val_shift_limit=18, p=1.0),
            A.ColorJitter(
                brightness=0.25, contrast=0.25, saturation=0.20, hue=0.06, p=1.0),
            A.CLAHE(clip_limit=(1, 4), p=1.0),
        ], p=0.9),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.1), p=1.0),
        ], p=0.25),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.12), p=1.0),
            A.ISONoise(p=1.0),
            A.ImageCompression(quality_range=(45, 90), p=1.0),
        ], p=0.25),
        A.RandomShadow(p=0.20),
        A.RandomFog(fog_coef_range=(0.1, 0.35), p=0.12),
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(12, 56), hole_width_range=(12, 56), p=0.35),
        A.ToGray(p=0.08),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])'''

old_tta = '''def get_tta_transforms(img_size=384):
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    crop = int(img_size * 0.9)
    return [
        # 0: base (center crop)
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
        # 1: hflip
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        # 2: brightness
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        # 3: tighter crop (90% of 90% = 81%)
        A.Compose([A.CenterCrop(int(img_size * 0.9), int(img_size * 0.9)),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
    ]'''

new_tta = '''def get_tta_transforms(img_size=384):
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    crop = int(img_size * 0.9)
    return [
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size), A.HorizontalFlip(p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.CLAHE(clip_limit=(1, 3), p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(crop, crop),
                    A.Resize(img_size, img_size),
                    A.Affine(scale=(0.97, 1.03), translate_percent=(-0.02, 0.02), rotate=(-3, 3), p=1.0),
                    A.Normalize(**norm), ToTensorV2()]),
        A.Compose([A.CenterCrop(int(img_size * 0.9), int(img_size * 0.9)),
                    A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()]),
    ]'''

if old_train in text:
    text = text.replace(old_train, new_train)
else:
    print('[WARN] train transform block not matched')

if old_tta in text:
    text = text.replace(old_tta, new_tta)
else:
    print('[WARN] tta transform block not matched')

path.write_text(text, encoding='utf-8')
print('코랩 런타임용 datasets.py 패치 완료')
print('  - train augmentation 강화')
print('  - TTA 6개 (전부 CenterCrop 90% 적용)')

In [ ]:
# 3. 데이터 압축 해제 (Drive data.zip → 로컬)
# Drive에서 폴더 복사는 너무 느림 → zip 해제가 훨씬 빠름

import os, shutil, zipfile, time

DRIVE_DIR  = "/content/drive/MyDrive/dacon"
WORK_DIR   = "/content/dacon"
LOCAL_DATA = os.path.join(WORK_DIR, "data")
ZIP_PATH   = os.path.join(DRIVE_DIR, "data.zip")

# 기존 data/ 정리
if os.path.islink(LOCAL_DATA):
    os.unlink(LOCAL_DATA)
elif os.path.isdir(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
os.makedirs(LOCAL_DATA, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print(f"[ERROR] {ZIP_PATH} 를 찾을 수 없습니다.")
    print(f"  Drive에 data.zip을 업로드하세요: {DRIVE_DIR}/data.zip")
    print(f"  zip 안에 open/, shapestacks/ 폴더가 있어야 합니다.")
else:
    zip_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f"data.zip 발견: {zip_mb:.0f} MB")
    print(f"해제 중... (로컬 SSD → 빠름)")

    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_DATA)
    elapsed = time.time() - t0
    print(f"  해제 완료: {elapsed:.1f}초")

# --- 검증 ---
for f in ["open/train.csv", "open/dev.csv", "open/sample_submission.csv"]:
    p = os.path.join(LOCAL_DATA, f)
    ok = "[OK]" if os.path.exists(p) else "[FAIL]"
    print(f"  {ok} {f}")

for split in ["train", "dev", "test"]:
    d = os.path.join(LOCAL_DATA, "open", split)
    if os.path.isdir(d):
        n = len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])
        print(f"  {split}/: {n}개")

ss = os.path.join(LOCAL_DATA, "shapestacks", "shapestacks", "recordings")
if os.path.exists(ss):
    h6 = [d for d in os.listdir(ss) if "h=6" in d]
    print(f"  ShapeStacks h=6: {len(h6)} scenarios")
else:
    print("  [SKIP] ShapeStacks 없음 (Dacon only)")

free = shutil.disk_usage('/content').free / 1e9
print(f"\n로컬 디스크 여유: {free:.1f} GB")

In [ ]:
# 5. 설정 검증
import os
import shutil

WORK_DIR  = "/content/dacon"
DRIVE_DIR = "/content/drive/MyDrive/dacon"

print("=" * 50)
print("  환경 검증")
print("=" * 50)

for f in ["models.py", "datasets.py", "train.py", "train_v2.py", "inference.py"]:
    ok = os.path.exists(os.path.join(WORK_DIR, f))
    print(f"  {'[OK]' if ok else '[FAIL]'} {f}")

tc = os.path.join(WORK_DIR, "data", "open", "train.csv")
print(f"  {'[OK]' if os.path.exists(tc) else '[FAIL]'} train.csv")

dc = os.path.join(WORK_DIR, "data", "open", "dev.csv")
print(f"  {'[OK]' if os.path.exists(dc) else '[FAIL]'} dev.csv")

ss = os.path.join(WORK_DIR, "data", "shapestacks", "shapestacks", "recordings")
if os.path.exists(ss):
    h6 = [d for d in os.listdir(ss) if "h=6" in d]
    print(f"  [OK] ShapeStacks h=6: {len(h6)} scenarios")
else:
    print(f"  [SKIP] ShapeStacks 없음 (Dacon only)")

# 체크포인트: 로컬 디렉토리 (Drive symlink 아님)
ckpt_dir = os.path.join(WORK_DIR, "checkpoints")
is_local = os.path.isdir(ckpt_dir) and not os.path.islink(ckpt_dir)
print(f"  {'[OK]' if is_local else '[WARN]'} checkpoints/ (로컬 저장)")

# submissions: Drive 심볼링크
sub_link = os.path.join(WORK_DIR, "submissions")
is_drive = os.path.islink(sub_link) and "drive" in os.readlink(sub_link)
print(f"  {'[OK]' if is_drive else '[FAIL]'} submissions/ -> Drive")

# 로컬 체크포인트 현황
ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pth')] if os.path.isdir(ckpt_dir) else []
resume_ckpts = [c for c in ckpts if "ckpt" in c]
best_ckpts   = [c for c in ckpts if "ckpt" not in c]

print()
if best_ckpts:
    print(f"  로컬 Best 가중치: {len(best_ckpts)}개")
    for c in sorted(best_ckpts):
        mb = os.path.getsize(os.path.join(ckpt_dir, c)) / 1e6
        print(f"    {c} ({mb:.0f} MB)")
if resume_ckpts:
    print(f"  로컬 Resume ckpt: {len(resume_ckpts)}개")
    for c in sorted(resume_ckpts):
        print(f"    {c}")
if not ckpts:
    print("  체크포인트 없음 -> 새로 학습 시작")

# Drive 체크포인트 현황
drive_ckpt = os.path.join(DRIVE_DIR, "checkpoints")
drive_ckpts = [f for f in os.listdir(drive_ckpt) if f.endswith('.pth')] if os.path.isdir(drive_ckpt) else []
if drive_ckpts:
    print(f"\n  Drive 백업: {len(drive_ckpts)}개")
    for c in sorted(drive_ckpts):
        mb = os.path.getsize(os.path.join(drive_ckpt, c)) / 1e6
        print(f"    {c} ({mb:.0f} MB)")

free = shutil.disk_usage('/content').free / 1e9
print(f"\n  디스크 여유: {free:.1f} GB")
print("=" * 50)

In [ ]:
# 6. 학습 설정 (v7: EVA-Giant 최적화 — LLRD + 보수적 LR + 강한 규제)
# 참고: EVA-Giant(1B params) fine-tuning 정석
#   - 층별 학습률 감소 (LLRD 0.9): 하위 층의 기초 특징 보호
#   - 낮은 LR (head 5e-5, bb 1e-5): catastrophic forgetting 방지
#   - 높은 Weight Decay (0.05): 과적합 방지
#   - Warmup 2 epoch: 학습 초반 안정화

BACKBONE = "eva_giant"
PRETRAIN_EPOCHS = 0
FINETUNE_EPOCHS = 20
PATIENCE = 15

MERGE_DEV = True
LOSS = "ce"
NO_MIXUP = True
HEAD_TYPE = "simple"
SIMPLE_AUG = True
SCHEDULER = "cosine"

# EVA-Giant 최적화 LR (기존 2e-4/2e-5 → 5e-5/1e-5로 보수적 설정)
HEAD_LR = 5e-5
BB_LR = 1e-5
WEIGHT_DECAY = 0.05      # 기존 0.01 → 0.05 (강한 규제)
DROP_RATE = 0.3

# 층별 학습률 감소 (Layer-wise LR Decay)
LAYER_DECAY = 0.9         # EVA-Giant 추천값 (Large=0.8, Base=0.7)
WARMUP_EPOCHS = 2         # Linear warmup 2 epoch

USE_VIDEO = False

import os
os.chdir("/content/dacon")

print(f"백본: {BACKBONE}")
print(f"Pretrain: {PRETRAIN_EPOCHS} ep -> Finetune: {FINETUNE_EPOCHS} ep")
print(f"Patience: {PATIENCE}")
print(f"  merge_dev={MERGE_DEV} | loss={LOSS} | no_mixup={NO_MIXUP}")
print(f"  head={HEAD_TYPE} | aug={'simple' if SIMPLE_AUG else 'full'} | sched={SCHEDULER}")
print(f"  head_lr={HEAD_LR} | bb_lr={BB_LR} | wd={WEIGHT_DECAY} | drop={DROP_RATE}")
print(f"  LLRD: layer_decay={LAYER_DECAY} | warmup={WARMUP_EPOCHS}ep")

pt_path = os.path.join("checkpoints", f"{BACKBONE}_pretrained.pth")
if os.path.exists(pt_path):
    mb = os.path.getsize(pt_path) / 1e6
    print(f"\n[INFO] 사전학습 체크포인트: {pt_path} ({mb:.0f} MB)")

for fold in range(5):
    best = os.path.join("checkpoints", f"{BACKBONE}_fold{fold}.pth")
    status = "완료" if os.path.exists(best) else "미완료"
    print(f"  Fold {fold}: {status}")

In [ ]:
# 7. Stage 1: Pretrain (ShapeStacks h=6 + Dacon)
# 경쟁 모드에서는 기본적으로 건너뜀

import os, sys, shlex, subprocess
os.chdir("/content/dacon")

if PRETRAIN_EPOCHS <= 0:
    print("경쟁 모드 기본값: pretrain 생략")
else:
    cmd = [
        sys.executable, "-u", "train.py",
        "--backbone", BACKBONE,
        "--stage", "pretrain",
        "--pretrain_epochs", str(PRETRAIN_EPOCHS),
        "--batch_size_override", "16",
        "--grad_checkpointing",
        "--resume",
        "--num_workers", "0",
    ]
    cmd_str = shlex.join(cmd)
    print(f"CMD: {cmd_str}\n" + "="*50)

    my_env = os.environ.copy()
    my_env["TQDM_FORCE_TTY"] = "1"

    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=my_env
    )

    while True:
        char = process.stdout.read(1)
        if not char and process.poll() is not None:
            break
        sys.stdout.write(char)
        sys.stdout.flush()

    process.wait()
    if process.returncode != 0:
        print(f"\n[ERROR] Pretrain 실패 (exit code {process.returncode})")


In [ ]:
# 8-0. Finetune Fold 0

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 0
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--layer_decay", str(LAYER_DECAY),
    "--warmup_epochs", str(WARMUP_EPOCHS),
    "--grad_checkpointing",
    "--resume",
    "--init_from_best",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)


In [ ]:
# 8-1. Finetune Fold 1

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 1
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--layer_decay", str(LAYER_DECAY),
    "--warmup_epochs", str(WARMUP_EPOCHS),
    "--grad_checkpointing",
    "--resume",
    "--init_from_best",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)


In [ ]:
# 8-2. Finetune Fold 2

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 2
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--layer_decay", str(LAYER_DECAY),
    "--warmup_epochs", str(WARMUP_EPOCHS),
    "--grad_checkpointing",
    "--resume",
    "--init_from_best",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)


In [ ]:
# 8-3. Finetune Fold 3

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 3
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--layer_decay", str(LAYER_DECAY),
    "--warmup_epochs", str(WARMUP_EPOCHS),
    "--grad_checkpointing",
    "--resume",
    "--init_from_best",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)


In [ ]:
# 8-4. Finetune Fold 4

import os, sys, shlex, subprocess, shutil
os.chdir("/content/dacon")

FOLD = 4
cmd = [
    sys.executable, "-u", "train.py",
    "--backbone", BACKBONE,
    "--stage", "finetune",
    "--finetune_epochs", str(FINETUNE_EPOCHS),
    "--patience", str(PATIENCE),
    "--fold", str(FOLD),
    "--loss", LOSS,
    "--scheduler", SCHEDULER,
    "--head_type", HEAD_TYPE,
    "--head_lr", str(HEAD_LR),
    "--bb_lr", str(BB_LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--drop_rate", str(DROP_RATE),
    "--layer_decay", str(LAYER_DECAY),
    "--warmup_epochs", str(WARMUP_EPOCHS),
    "--grad_checkpointing",
    "--resume",
    "--init_from_best",
    "--num_workers", "0",
]

if MERGE_DEV:
    cmd += ["--merge_dev"]
if NO_MIXUP:
    cmd += ["--no_mixup"]
if SIMPLE_AUG:
    cmd += ["--simple_aug"]

if USE_VIDEO:
    cmd += ["--batch_size_override", "4", "--use_video"]
else:
    cmd += ["--batch_size_override", "16"]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()
if process.returncode != 0:
    print(f"\n[ERROR] Fold {FOLD} 실패 (exit code {process.returncode})")
else:
    free = shutil.disk_usage('/content').free / 1e9
    print(f"\n디스크 여유: {free:.1f} GB")
    print("Drive 동기화...")
    sync_best_to_drive(BACKBONE)


In [ ]:
# 9. 추론 + 제출 파일 생성 (inference_v2: dual-crop + checkpoint selection)

import os, sys, shlex, subprocess, glob
import pandas as pd
os.chdir("/content/dacon")

# --- v2 설정 ---
# 방법 1: 특정 체크포인트 파일 직접 지정 (vfa 등 이름이 다른 경우)
USE_CHECKPOINT = None   # None이면 USE_FOLDS 사용. 예: "eva_giant_fold0.pth"
# 방법 2: fold 번호로 지정 (eva_giant_fold{N}.pth 패턴)
USE_FOLDS = [0, 1, 2, 3, 4]

FRONT_CROP = 0.9               # Front view CenterCrop 비율
TOP_CROP = 0.7                 # Top view CenterCrop 비율 (배경 제거 강화)

cmd = [
    sys.executable, "-u", "inference_v2.py",
    "--backbones", BACKBONE,
    "--head_type", HEAD_TYPE,
    "--tta",
    "--temperature", "1.0",
    "--front_crop", str(FRONT_CROP),
    "--top_crop", str(TOP_CROP),
    "--num_workers", "0",
]

if USE_CHECKPOINT:
    ckpts = USE_CHECKPOINT if isinstance(USE_CHECKPOINT, list) else [USE_CHECKPOINT]
    cmd += ["--checkpoint", *ckpts]
else:
    cmd += ["--folds", *[str(f) for f in USE_FOLDS]]

cmd_str = shlex.join(cmd)
print(f"CMD: {cmd_str}\n" + "="*50)

my_env = os.environ.copy()
my_env["TQDM_FORCE_TTY"] = "1"

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=my_env
)

while True:
    char = process.stdout.read(1)
    if not char and process.poll() is not None:
        break
    sys.stdout.write(char)
    sys.stdout.flush()

process.wait()

if process.returncode != 0:
    print(f"\n[ERROR] 추론 실패 (exit code {process.returncode})")
else:
    subs = sorted(glob.glob("/content/dacon/submissions/*_v2.csv"))
    if subs:
        latest = subs[-1]
        df = pd.read_csv(latest)
        print(f"\n최신 제출: {os.path.basename(latest)}")
        print(f"  샘플 수: {len(df)}")
        print(f"  unstable_prob: mean={df['unstable_prob'].mean():.4f}, std={df['unstable_prob'].std():.4f}")
        print(df.head())
        print(f"\n제출 파일은 Drive에 저장됨: /content/drive/MyDrive/dacon/submissions/")
    else:
        print("\n제출 파일이 생성되지 않았습니다.")

In [ ]:
# 11. Test 제출 파일 검증
import pandas as pd

df = pd.read_csv('submission_eva_giant_3seeds_tta.csv')

unstable_mean = df['unstable_prob'].mean()
unstable_std = df['unstable_prob'].std()

print(f"unstable_prob: mean={unstable_mean:.4f}, std={unstable_std:.4f}")
print(df[['id', 'unstable_prob', 'stable_prob']].head().to_string())

In [ ]:
# 11. 모든 체크포인트 모델별 Train/Dev Loss 비교
# checkpoints/ 안의 모든 .pth 파일을 하나씩 로드하여 train/dev logloss 계산

import torch, os, cv2, numpy as np, pandas as pd
import torch.nn.functional as TF
from torch.amp import autocast
from sklearn.metrics import log_loss as sk_logloss
import albumentations as A
from albumentations.pytorch import ToTensorV2

os.chdir("/content/dacon")
from models import build_model, get_backbone_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ========== 데이터 로드 함수 ==========
def load_split(csv_path, data_dir, img_size, crop_ratio=0.9):
    """CSV + 이미지 → (front_tensors, top_tensors, labels)"""
    df = pd.read_csv(csv_path)
    norm = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    crop = int(img_size * crop_ratio)
    tf = A.Compose([
        A.CenterCrop(crop, crop), A.Resize(img_size, img_size),
        A.Normalize(**norm), ToTensorV2(),
    ])
    fronts, tops, labels = [], [], []
    for _, row in df.iterrows():
        sid = row["id"]
        d = os.path.join(data_dir, sid)
        fr = cv2.cvtColor(cv2.imread(os.path.join(d, "front.png")), cv2.COLOR_BGR2RGB)
        tp = cv2.cvtColor(cv2.imread(os.path.join(d, "top.png")), cv2.COLOR_BGR2RGB)
        fronts.append(tf(image=fr)["image"])
        tops.append(tf(image=tp)["image"])
        labels.append(1 if row["label"] == "unstable" else 0)
    return torch.stack(fronts), torch.stack(tops), np.array(labels)


def evaluate_model(model, fronts, tops, labels, device, batch_size=32):
    """모델의 logloss, accuracy, per-class accuracy 계산"""
    model.eval()
    all_probs = []
    n = len(labels)
    with torch.no_grad():
        for i in range(0, n, batch_size):
            fr = fronts[i:i+batch_size].to(device)
            tp = tops[i:i+batch_size].to(device)
            with autocast("cuda"):
                logits = model(fr, tp)
            probs = TF.softmax(logits.float(), dim=1).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.concatenate(all_probs)

    eps = 1e-7
    clipped = np.clip(all_probs, eps, 1 - eps)
    logloss = sk_logloss(labels, clipped, labels=[0, 1])
    preds = all_probs.argmax(axis=1)
    acc = (preds == labels).mean()

    unstable_mask = labels == 1
    stable_mask = labels == 0
    unstable_acc = (preds[unstable_mask] == 1).mean() if unstable_mask.sum() > 0 else 0
    stable_acc = (preds[stable_mask] == 0).mean() if stable_mask.sum() > 0 else 0

    return {
        "logloss": logloss, "accuracy": acc,
        "unstable_acc": unstable_acc, "stable_acc": stable_acc,
        "mean_p_unstable": all_probs[:, 1].mean(),
    }


# ========== 데이터 준비 ==========
cfg = get_backbone_config(BACKBONE)
img_size = cfg["img_size"]

print("Loading train data...")
train_fronts, train_tops, train_labels = load_split(
    "data/open/train.csv", "data/open/train", img_size)
print(f"  Train: {len(train_labels)} samples "
      f"(unstable={train_labels.sum()}, stable={(1-train_labels).sum()})")

print("Loading dev data...")
dev_fronts, dev_tops, dev_labels = load_split(
    "data/open/dev.csv", "data/open/dev", img_size)
print(f"  Dev: {len(dev_labels)} samples "
      f"(unstable={dev_labels.sum()}, stable={(1-dev_labels).sum()})")

# ========== 모든 체크포인트 평가 ==========
ckpt_dir = "checkpoints"
all_ckpts = sorted([f for f in os.listdir(ckpt_dir)
                    if f.endswith('.pth') and 'ckpt' not in f and 'pretrain' not in f])

print(f"\n{'='*90}")
print(f"  {'Checkpoint':<35} {'Train Loss':>12} {'Dev Loss':>12} {'Train Acc':>10} {'Dev Acc':>10}")
print(f"{'='*90}")

results = []
for ckpt_name in all_ckpts:
    ckpt_path = os.path.join(ckpt_dir, ckpt_name)

    model = build_model(BACKBONE, pretrained=False, num_classes=2,
                        drop_rate=0.0, head_type=HEAD_TYPE)
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    model_sd = model.state_dict()
    filtered = {k: v for k, v in sd.items()
                if k in model_sd and v.shape == model_sd[k].shape}
    model.load_state_dict(filtered, strict=False)
    model = model.to(device).eval()

    train_res = evaluate_model(model, train_fronts, train_tops, train_labels, device)
    dev_res = evaluate_model(model, dev_fronts, dev_tops, dev_labels, device)

    results.append({
        "checkpoint": ckpt_name,
        "train_logloss": train_res["logloss"], "dev_logloss": dev_res["logloss"],
        "train_acc": train_res["accuracy"], "dev_acc": dev_res["accuracy"],
        "train_unstable_acc": train_res["unstable_acc"], "train_stable_acc": train_res["stable_acc"],
        "dev_unstable_acc": dev_res["unstable_acc"], "dev_stable_acc": dev_res["stable_acc"],
    })

    print(f"  {ckpt_name:<35} {train_res['logloss']:>12.7f} {dev_res['logloss']:>12.7f} "
          f"{train_res['accuracy']:>9.4f} {dev_res['accuracy']:>9.4f}")

    del model
    torch.cuda.empty_cache()

# ========== 상세 결과 ==========
print(f"\n{'='*110}")
print(f"  {'Checkpoint':<35} {'Train LL':>11} {'Dev LL':>11} "
      f"{'T-Acc':>7} {'D-Acc':>7} "
      f"{'T-Unst':>7} {'T-Stab':>7} {'D-Unst':>7} {'D-Stab':>7}")
print(f"  {'-'*103}")

for r in results:
    print(f"  {r['checkpoint']:<35} "
          f"{r['train_logloss']:>11.7f} {r['dev_logloss']:>11.7f} "
          f"{r['train_acc']:>6.4f} {r['dev_acc']:>6.4f} "
          f"{r['train_unstable_acc']:>6.4f} {r['train_stable_acc']:>6.4f} "
          f"{r['dev_unstable_acc']:>6.4f} {r['dev_stable_acc']:>6.4f}")

# ========== Best 추천 ==========
best_dev = min(results, key=lambda x: x["dev_logloss"])
best_train = min(results, key=lambda x: x["train_logloss"])

print(f"\n{'='*70}")
print(f"  Best Dev LogLoss:   {best_dev['checkpoint']:<35} -> {best_dev['dev_logloss']:.7f}")
print(f"  Best Train LogLoss: {best_train['checkpoint']:<35} -> {best_train['train_logloss']:.7f}")
print(f"{'='*70}")
print(f"\n  [TIP] Dev LL이 낮은 모델을 선택 | Train LL만 낮으면 = 과적합")

In [ ]:
# 12. Attention Rollout 시각화 (ViT 전용)
# EVA-giant: EvaAttention은 F.linear로 QKV를 직접 처리 → blk.attn에 hook 필요

import torch, torch.nn as nn, torch.nn.functional as F
import os, cv2, numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- 한글 폰트 ---
import subprocess, sys
_noto = "/usr/share/fonts/truetype/noto/NotoSansKR-Regular.otf"
if not os.path.exists(_noto):
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], check=False)
    fm._load_fontmanager(try_read_cache=False)
for f in fm.findSystemFonts():
    if "NotoSansCJK" in f or "NotoSansKR" in f:
        fm.fontManager.addfont(f)
        plt.rcParams["font.family"] = fm.FontProperties(fname=f).get_name()
        break
plt.rcParams["axes.unicode_minus"] = False

os.chdir("/content/dacon")
from models import build_model, get_backbone_config

# ========== 설정 ==========
# 체크포인트 직접 지정 (None이면 BACKBONE + VIS_FOLD에서 자동 생성)
VIS_CHECKPOINT = None   # 예: "eva_giant_fold0.pth"
VIS_FOLD = 0            # VIS_CHECKPOINT=None일 때만 사용
N_PER_GROUP = 4
PATCH_SIZE = 14
VIS_FRONT_CROP = 0.9    # Front view CenterCrop 비율 (inference_v2와 맞춤)
VIS_TOP_CROP = 0.7      # Top view CenterCrop 비율 (inference_v2와 맞춤)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = get_backbone_config(BACKBONE)
img_size = cfg["img_size"]      # 336
grid_size = img_size // PATCH_SIZE  # 24
front_crop = int(img_size * VIS_FRONT_CROP)
top_crop = int(img_size * VIS_TOP_CROP)

# ========== 모델 로드 ==========
model = build_model(BACKBONE, pretrained=False, num_classes=2,
                    drop_rate=0.0, head_type=HEAD_TYPE)
if VIS_CHECKPOINT:
    ckpt_path = os.path.join("checkpoints", VIS_CHECKPOINT)
else:
    ckpt_path = os.path.join("checkpoints", f"{BACKBONE}_fold{VIS_FOLD}.pth")
assert os.path.exists(ckpt_path), f"체크포인트 없음: {ckpt_path}"
sd = torch.load(ckpt_path, map_location="cpu", weights_only=True)
model_sd = model.state_dict()
filtered = {k: v for k, v in sd.items()
            if k in model_sd and v.shape == model_sd[k].shape}
model.load_state_dict(filtered, strict=False)
model = model.to(device).eval()
num_blocks = len(model.backbone.blocks)
print(f"Model: {ckpt_path}")
print(f"  {num_blocks} blocks, grid={grid_size}x{grid_size}, patch={PATCH_SIZE}")
print(f"  Front CenterCrop: {VIS_FRONT_CROP} ({front_crop}px) | Top CenterCrop: {VIS_TOP_CROP} ({top_crop}px)")

# ========== Transform (Front/Top 별도 crop) ==========
norm_args = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
front_val_tf = A.Compose([
    A.CenterCrop(front_crop, front_crop), A.Resize(img_size, img_size),
    A.Normalize(**norm_args), ToTensorV2(),
])
top_val_tf = A.Compose([
    A.CenterCrop(top_crop, top_crop), A.Resize(img_size, img_size),
    A.Normalize(**norm_args), ToTensorV2(),
])
front_disp_tf = A.Compose([A.CenterCrop(front_crop, front_crop), A.Resize(img_size, img_size)])
top_disp_tf = A.Compose([A.CenterCrop(top_crop, top_crop), A.Resize(img_size, img_size)])

# ========== Attention 추출 (EvaAttention 대응) ==========
def extract_attentions(backbone, img_tensor):
    """EvaAttention에서 attention weight 추출."""
    attns = []
    hooks = []

    for blk in backbone.blocks:
        am = blk.attn

        def hook_fn(module, inputs, output):
            x = inputs[0]
            B, N, C = x.shape

            if module.qkv is not None:
                q_bias = getattr(module, 'q_bias', None)
                if q_bias is not None:
                    k_bias = getattr(module, 'k_bias', torch.zeros_like(q_bias))
                    v_bias = getattr(module, 'v_bias', torch.zeros_like(q_bias))
                    qkv_bias = torch.cat((q_bias, k_bias, v_bias))
                    qkv = F.linear(x, weight=module.qkv.weight, bias=qkv_bias)
                else:
                    qkv = F.linear(x, weight=module.qkv.weight, bias=module.qkv.bias)
                qkv = qkv.reshape(B, N, 3, module.num_heads, -1).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv.unbind(0)
            else:
                q = module.q_proj(x).reshape(B, N, module.num_heads, -1).transpose(1, 2)
                k = module.k_proj(x).reshape(B, N, module.num_heads, -1).transpose(1, 2)

            q = module.q_norm(q)
            k = module.k_norm(k)
            a = (q @ k.transpose(-2, -1)) * module.scale
            attns.append(a.softmax(dim=-1).detach().cpu())

        hooks.append(am.register_forward_hook(hook_fn))

    with torch.no_grad():
        backbone(img_tensor)
    for h in hooks:
        h.remove()
    return attns


def attention_rollout(attns):
    """Attention Rollout: residual 포함 layer간 누적"""
    N = attns[0].shape[-1]
    result = torch.eye(N).unsqueeze(0)
    for attn in attns:
        a = attn.mean(dim=1)
        a = 0.5 * a + 0.5 * torch.eye(N).unsqueeze(0)
        result = a @ result
    cls = result[0, 0, 1:].numpy()
    return cls / (cls.max() + 1e-8)


def last_layer_attention(attns):
    """마지막 layer의 CLS attention"""
    a = attns[-1].mean(dim=1)
    cls = a[0, 0, 1:].numpy()
    return cls / (cls.max() + 1e-8)


def attn_to_heatmap(attn_vec, grid_size, img_size):
    h = attn_vec.reshape(grid_size, grid_size)
    h = cv2.resize(h.astype(np.float32), (img_size, img_size),
                   interpolation=cv2.INTER_CUBIC)
    return np.clip(h, 0, 1)

# ========== 디버그: hook 동작 확인 ==========
_test_img = torch.randn(1, 3, img_size, img_size, device=device)
_test_attns = extract_attentions(model.backbone, _test_img)
print(f"  Hook test: {len(_test_attns)} layers captured "
      f"(expected {num_blocks}), shape={_test_attns[0].shape if _test_attns else 'EMPTY'}")
assert len(_test_attns) == num_blocks, f"Hook failed! Got {len(_test_attns)} attns"
del _test_img, _test_attns
torch.cuda.empty_cache()

# ========== 전체 테스트 예측 ==========
test_dir = os.path.join("data", "open", "test")
sub_df = pd.read_csv(os.path.join("data", "open", "sample_submission.csv"))
all_ids = sub_df["id"].tolist()

preds = {}
print(f"Predicting {len(all_ids)} test samples...")
with torch.no_grad():
    for sid in all_ids:
        d = os.path.join(test_dir, sid)
        fr = cv2.cvtColor(cv2.imread(os.path.join(d, "front.png")), cv2.COLOR_BGR2RGB)
        tp = cv2.cvtColor(cv2.imread(os.path.join(d, "top.png")), cv2.COLOR_BGR2RGB)
        fr_t = front_val_tf(image=fr)["image"].unsqueeze(0).to(device)
        tp_t = top_val_tf(image=tp)["image"].unsqueeze(0).to(device)
        logits = model(fr_t, tp_t)
        probs = torch.softmax(logits, dim=1)[0]
        preds[sid] = float(probs[1])

sorted_items = sorted(preds.items(), key=lambda x: x[1], reverse=True)
n_unstable = sum(1 for _, p in sorted_items if p > 0.5)
print(f"  Unstable: {n_unstable} | Stable: {len(all_ids) - n_unstable}")

# ========== 히스토그램 ==========
ckpt_label = VIS_CHECKPOINT or f"{BACKBONE}_fold{VIS_FOLD}"
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(list(preds.values()), bins=50, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(0.5, color="red", ls="--", lw=1.5, label="threshold")
ax.set_xlabel("P(unstable)"); ax.set_ylabel("Count")
ax.set_title(f"Test Prediction Distribution - {ckpt_label}\n"
             f"Front crop={VIS_FRONT_CROP} | Top crop={VIS_TOP_CROP}")
ax.legend(); plt.tight_layout(); plt.show()

# ========== 샘플 선택 ==========
top_unstable = [sid for sid, _ in sorted_items[:N_PER_GROUP]]
top_stable   = [sid for sid, _ in sorted_items[-N_PER_GROUP:]]
mid = len(sorted_items) // 2
borderline   = [sid for sid, _ in sorted_items[mid:mid+N_PER_GROUP]]

groups = [
    ("UNSTABLE (high confidence)", top_unstable),
    ("BORDERLINE (~50%)", borderline),
    ("STABLE (high confidence)", top_stable),
]

# ========== Attention 시각화 ==========
print("\nAttention Rollout + Last-Layer Attention 추출 중...")

for gname, ids in groups:
    n = len(ids)
    fig, axes = plt.subplots(3, 2 * n, figsize=(4.5 * n, 7.5))
    fig.suptitle(f"{gname}  [{ckpt_label}]", fontsize=14, fontweight="bold", y=1.01)

    for j, sid in enumerate(ids):
        d = os.path.join(test_dir, sid)
        fr_raw = cv2.cvtColor(cv2.imread(os.path.join(d, "front.png")), cv2.COLOR_BGR2RGB)
        tp_raw = cv2.cvtColor(cv2.imread(os.path.join(d, "top.png")), cv2.COLOR_BGR2RGB)
        fr_disp = front_disp_tf(image=fr_raw)["image"]
        tp_disp = top_disp_tf(image=tp_raw)["image"]
        fr_t = front_val_tf(image=fr_raw)["image"].unsqueeze(0).to(device)
        tp_t = top_val_tf(image=tp_raw)["image"].unsqueeze(0).to(device)

        p = preds[sid]

        fr_attns = extract_attentions(model.backbone, fr_t)
        fr_rollout = attn_to_heatmap(attention_rollout(fr_attns), grid_size, img_size)
        fr_last    = attn_to_heatmap(last_layer_attention(fr_attns), grid_size, img_size)

        tp_attns = extract_attentions(model.backbone, tp_t)
        tp_rollout = attn_to_heatmap(attention_rollout(tp_attns), grid_size, img_size)
        tp_last    = attn_to_heatmap(last_layer_attention(tp_attns), grid_size, img_size)

        cf, ct = j * 2, j * 2 + 1

        axes[0, cf].imshow(fr_disp); axes[0, cf].imshow(fr_rollout, alpha=0.6, cmap="inferno")
        axes[0, cf].set_title(f"{sid}\nP={p:.4f} | Front Rollout", fontsize=7)
        axes[0, cf].axis("off")
        axes[0, ct].imshow(tp_disp); axes[0, ct].imshow(tp_rollout, alpha=0.6, cmap="inferno")
        axes[0, ct].set_title(f"Top Rollout (crop={VIS_TOP_CROP})", fontsize=7)
        axes[0, ct].axis("off")

        axes[1, cf].imshow(fr_disp); axes[1, cf].imshow(fr_last, alpha=0.6, cmap="inferno")
        axes[1, cf].set_title(f"Front Last-Layer", fontsize=7)
        axes[1, cf].axis("off")
        axes[1, ct].imshow(tp_disp); axes[1, ct].imshow(tp_last, alpha=0.6, cmap="inferno")
        axes[1, ct].set_title(f"Top Last-Layer", fontsize=7)
        axes[1, ct].axis("off")

        axes[2, cf].imshow(fr_disp)
        axes[2, cf].set_title(f"Front (crop={VIS_FRONT_CROP})", fontsize=7)
        axes[2, cf].axis("off")
        axes[2, ct].imshow(tp_disp)
        axes[2, ct].set_title(f"Top (crop={VIS_TOP_CROP})", fontsize=7)
        axes[2, ct].axis("off")

    plt.tight_layout()
    plt.show()

# ========== 요약 ==========
print("\n" + "=" * 60)
print(f"  [{ckpt_label}] Top 10 Unstable")
print("=" * 60)
for sid, p in sorted_items[:10]:
    print(f"  {sid}: P(unstable)={p:.6f}")
print(f"\n  Top 10 Stable")
print("-" * 60)
for sid, p in sorted_items[-10:]:
    print(f"  {sid}: P(unstable)={p:.6f}")

print(f"\n[INFO] Rollout = 전체 {num_blocks} layer attention 누적 (전역적 시야)")
print(f"[INFO] Last-Layer = 마지막 layer만 (최종 판단 근거)")
print(f"[INFO] Front crop={VIS_FRONT_CROP} ({front_crop}px) | Top crop={VIS_TOP_CROP} ({top_crop}px)")
print(f"[INFO] 블록 영역에 밝으면 OK / 배경에 밝으면 모델이 배경에 의존")

In [ ]:
# 13. 예측값 날카롭게 보정 (Power Sharpening → 0/1에 가깝게)
# 모델이 0.4~0.6 중간값에 몰릴 때 분포를 0/1로 밀어붙임
# alpha > 1: 더 날카롭게 | alpha = 1.0: 변화 없음

import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.chdir("/content/dacon")


def power_sharpen(p_unstable, alpha=2.0):
    """Binary power sharpening: p_sharp = p^α / (p^α + (1-p)^α)

    alpha=1.0 → 변화 없음 (원본)
    alpha=1.5 → 약한 날카로움
    alpha=2.0 → 중간 날카로움 (기본값)
    alpha=3.0 → 강한 날카로움
    alpha=4.0 → 매우 날카로움 (0.7+ → 거의 1.0, 0.3- → 거의 0.0)
    """
    p = np.clip(p_unstable, 1e-7, 1.0 - 1e-7)
    q = 1.0 - p
    p_a = p ** alpha
    q_a = q ** alpha
    return p_a / (p_a + q_a)


# ── 1. 최신 제출 파일 찾기 ────────────────────────────────────────────────
subs = sorted(glob.glob("/content/dacon/submissions/*.csv"))
if not subs:
    print("[ERROR] submissions/ 폴더에 CSV 없음 → 추론 먼저 실행 (셀 9)")
else:
    latest = subs[-1]
    df = pd.read_csv(latest)
    p_orig = df["unstable_prob"].values.copy()

    print(f"원본 파일: {os.path.basename(latest)}")
    print(f"  샘플 수: {len(df)}")
    print(f"  unstable_prob: mean={p_orig.mean():.4f}, std={p_orig.std():.4f}")
    print(f"  > 0.5: {(p_orig > 0.5).sum()}개  |  < 0.5: {(p_orig < 0.5).sum()}개")

    # ── 2. 여러 alpha 비교 ─────────────────────────────────────────────────
    ALPHA = 2.0   # 최종 적용할 alpha (변경 가능)
    alphas = [1.5, 2.0, 3.0, 4.0]

    fig, axes = plt.subplots(1, len(alphas) + 1, figsize=(4.5 * (len(alphas) + 1), 4))
    axes[0].hist(p_orig, bins=50, color="steelblue", edgecolor="white", alpha=0.85)
    axes[0].axvline(0.5, color="red", ls="--", lw=1.5)
    axes[0].set_title(f"원본\nmean={p_orig.mean():.3f}  std={p_orig.std():.3f}")
    axes[0].set_xlabel("P(unstable)")
    axes[0].set_xlim(0, 1)

    for i, alpha in enumerate(alphas):
        p_sharp = power_sharpen(p_orig, alpha)
        color = "orange" if alpha == ALPHA else "tomato"
        axes[i + 1].hist(p_sharp, bins=50, color=color, edgecolor="white", alpha=0.85)
        axes[i + 1].axvline(0.5, color="navy", ls="--", lw=1.5)
        label = " ← 선택" if alpha == ALPHA else ""
        axes[i + 1].set_title(f"α={alpha}{label}\n"
                               f"mean={p_sharp.mean():.3f}  std={p_sharp.std():.3f}")
        axes[i + 1].set_xlabel("P(unstable)")
        axes[i + 1].set_xlim(0, 1)

    plt.suptitle("Power Sharpening 비교  (α가 클수록 0/1에 가깝게)", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── 3. 선택한 alpha로 저장 ─────────────────────────────────────────────
    p_final = power_sharpen(p_orig, ALPHA)
    df_sharp = df.copy()
    df_sharp["unstable_prob"] = p_final
    df_sharp["stable_prob"]   = 1.0 - p_final

    base = os.path.splitext(os.path.basename(latest))[0]
    out_name = f"{base}_sharp_a{str(ALPHA).replace('.', '')}.csv"
    out_path = f"/content/dacon/submissions/{out_name}"
    df_sharp.to_csv(out_path, index=False)

    print(f"\n저장 완료: {out_name}")
    print(f"  unstable_prob 보정 후: mean={p_final.mean():.4f}, std={p_final.std():.4f}")
    print(f"  > 0.5: {(p_final > 0.5).sum()}개  |  < 0.5: {(p_final < 0.5).sum()}개")
    print(f"\n[TIP] ALPHA 변수를 바꾸면 날카로움 조절 가능")
    print("  α=1.5: 약한  α=2.0: 기본  α=3.0: 강함  α=4.0: 매우 강함")
